# Energy-Aware Production Scheduling
### Kapan pabrik baja paling boros listrik, dan bagaimana jadwal produksi bisa membantu?

**Penulis:** Muhammad Refaldi  |  **Dataset:** Steel Industry Energy Consumption (UCI Machine Learning Repository)

> Notebook ini adalah **kerangka**. Bagian bertanda `✍️ TULIS INSIGHT ANDA` sengaja dikosongkan: isi dengan temuan Anda sendiri setelah menjalankan sel di atasnya pada data asli.

## 1. Latar Belakang & Tujuan
Energi adalah komponen biaya besar di manufaktur berat. Jadwal produksi yang tidak memperhitungkan profil beban listrik dapat menaikkan biaya beban puncak, memicu penalti faktor daya, dan menambah emisi CO₂.

**Business questions**
1. Kapan konsumsi energi tertinggi (jam, hari)?
2. Seberapa besar porsi beban Light, Medium, dan Maximum?
3. Kapan power factor sering rendah?
4. Apa beda pola weekday dan weekend?
5. Apa peluang menggeser jadwal produksi agar beban lebih merata?

## 2. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})

# === UBAH BAGIAN INI ===
DATA_PATH = "steel_energy.csv"      # path ke CSV asli dari UCI
PF_THRESHOLD = 90                   # ambang power factor rendah (ASUMSI: sesuaikan dengan ketentuan tarif pabrik)
# Nama kolom: cocokkan dengan hasil df.columns di langkah berikutnya
COL_DATE, COL_USAGE = "date", "Usage_kWh"
COL_PF = "Lagging_Current_Power_Factor"
COL_WEEK, COL_DAY, COL_LOAD = "WeekStatus", "Day_of_week", "Load_Type"
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

## 3. Load & Inspeksi Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Ukuran data:", df.shape)
print("Kolom:", list(df.columns))
df.head()

In [ ]:
needed = [COL_DATE, COL_USAGE, COL_PF, COL_WEEK, COL_DAY, COL_LOAD]
missing_cols = [c for c in needed if c not in df.columns]
assert not missing_cols, f"Kolom tidak ditemukan: {missing_cols}. Sesuaikan nama kolom di sel Setup dengan df.columns."
df.info()

In [ ]:
df.describe().T

## 4. Data Cleaning & Persiapan

In [ ]:
# Missing value & duplikat
na = df.isna().sum()
print("Missing value per kolom:\n", na[na > 0])
print("Baris duplikat:", df.duplicated().sum())

# Parsing tanggal (format dd/mm/yyyy hh:mm) & fitur turunan
df[COL_DATE] = pd.to_datetime(df[COL_DATE], dayfirst=True)
df["hour"] = df[COL_DATE].dt.hour
df["low_pf"] = df[COL_PF] < PF_THRESHOLD
print("Rentang waktu:", df[COL_DATE].min(), "→", df[COL_DATE].max())

**Catatan cleaning:** ✍️ TULIS KEPUTUSAN ANDA (misalnya: ada/tidak missing value, bagaimana menangani duplikat atau outlier, dan alasannya).

## 5. Analisis

### 5.1 Profil konsumsi per jam, weekday vs weekend  *(BQ 1 & 4)*

In [ ]:
hourly = df.groupby([COL_WEEK, "hour"])[COL_USAGE].mean().unstack(0)
ax = hourly.plot(marker="o")
ax.set(title="Rata-rata konsumsi energi per jam (kWh / 15 menit)", xlabel="Jam", ylabel="kWh")
plt.tight_layout(); plt.show()
hourly.round(1).T

**Insight:** ✍️ TULIS INSIGHT ANDA (jam puncak berapa? berapa kali lipat dibanding jam sepi? bedanya dengan akhir pekan?)

### 5.2 Heatmap hari × jam  *(BQ 1)*

In [ ]:
heat = df.pivot_table(index=COL_DAY, columns="hour", values=COL_USAGE, aggfunc="mean")
heat = heat.reindex([d for d in DAY_ORDER if d in heat.index])
fig, ax = plt.subplots(figsize=(10, 4.2))
im = ax.imshow(heat, aspect="auto", cmap="YlOrRd")
ax.set_yticks(range(len(heat))); ax.set_yticklabels(heat.index)
ax.set_xticks(range(0, 24, 2)); ax.set(title="Rata-rata kWh: hari × jam", xlabel="Jam")
ax.grid(False); fig.colorbar(im, label="kWh"); plt.tight_layout(); plt.show()

**Insight:** ✍️ TULIS INSIGHT ANDA

### 5.3 Komposisi jenis beban  *(BQ 2)*

In [ ]:
share = pd.crosstab(df[COL_WEEK], df[COL_LOAD], normalize="index") * 100
ax = share.plot(kind="barh", stacked=True)
ax.set(title="Komposisi jenis beban (% interval)", xlabel="%", ylabel="")
plt.tight_layout(); plt.show()

# Porsi ENERGI (bukan hanya jumlah interval) per jenis beban
energy_share = (df.groupby(COL_LOAD)[COL_USAGE].sum() / df[COL_USAGE].sum() * 100).round(1)
print("Porsi energi total per jenis beban (%):\n", energy_share)
share.round(1)

**Insight:** ✍️ TULIS INSIGHT ANDA (berapa persen energi datang dari beban Maximum? kapan terjadi?)

### 5.4 Power factor rendah  *(BQ 3)*

In [ ]:
lowpf = df.groupby("hour")["low_pf"].mean() * 100
ax = lowpf.plot(kind="bar", color="#7f7f7f")
ax.set(title=f"% interval dengan power factor < {PF_THRESHOLD}, per jam", xlabel="Jam", ylabel="%")
plt.tight_layout(); plt.show()
print(f"Porsi interval PF < {PF_THRESHOLD}: {df['low_pf'].mean()*100:.1f}%")

**Insight:** ✍️ TULIS INSIGHT ANDA (di jam berapa PF rendah? apa hubungannya dengan beban?)

### 5.5 Skenario penggeseran beban  *(BQ 5, opsional)*

In [ ]:
# Skenario sederhana: sebagian energi beban Maximum di weekday digeser ke jam beban ringan.
SHIFT_PCT = 0.10          # ASUMSI: 10% energi Maximum digeser (ubah sesuai kelayakan operasional)
TARIFF_PEAK = None        # Rp/kWh jam puncak     (isi dari tarif pabrik)
TARIFF_OFFPEAK = None     # Rp/kWh jam non-puncak (isi dari tarif pabrik)

max_weekday = df[(df[COL_LOAD] == "Maximum_Load") & (df[COL_WEEK] == "Weekday")][COL_USAGE].sum()
shifted = max_weekday * SHIFT_PCT
print(f"Energi beban Maximum weekday: {max_weekday:,.0f} kWh")
print(f"Energi yang digeser ({SHIFT_PCT:.0%}): {shifted:,.0f} kWh")
if TARIFF_PEAK is not None and TARIFF_OFFPEAK is not None:
    print(f"Estimasi selisih biaya: Rp {(TARIFF_PEAK - TARIFF_OFFPEAK) * shifted:,.0f}")
else:
    print("Isi TARIFF_PEAK dan TARIFF_OFFPEAK untuk menghitung estimasi biaya.")

## 6. Insight & Rekomendasi

| Temuan | Dampak bisnis | Rekomendasi aksi |
|---|---|---|
| ✍️ | ✍️ | ✍️ |
| ✍️ | ✍️ | ✍️ |
| ✍️ | ✍️ | ✍️ |

## 7. Keterbatasan & Langkah Lanjutan
✍️ TULIS dengan jujur: misalnya data satu pabrik dan satu tahun, tidak ada data tarif atau jadwal produksi aktual, ambang PF merupakan asumsi. Langkah lanjutan: dashboard Tableau/Power BI, forecasting beban, validasi dengan data lapangan.